# Partial-Order Planner (POP) — AIMA-Style with Multi-Agent Extension

A **Partial-Order Planner** searches through the *space of plans* rather than the space of states.  
It follows the **least-commitment** strategy: actions and variable bindings are added only when required.

This notebook re-implements the AIMA `PartialOrderPlanner` and extends it with  
**multi-agent support** — a centralized `CoordAgent` computes the global POP, then the plan is  
distributed to individual agents (*Centralized Planning / Distributed Execution* — MAS Ch. 3).

---
**Plan data-structure** (AIMA §11.2):

| Component | Meaning |
|-----------|---------|
| `steps` | Set of actions including dummy `START` and `FINISH` |
| `orderings` | Set of constraints *A ≺ B* ("A before B") |
| `causal_links` | Set of links *A —p→ B* ("A achieves p for B") |
| `agenda` | Set of open preconditions `(step, literal)` |


## 1. Core Data Structures (AIMA §11.2)

In [1]:
from copy import deepcopy
from collections import defaultdict, deque
from itertools import count

# ─── Action ──────────────────────────────────────────────────────────────────
class Action:
    """
    A planning action, extended with an *agent* attribute for multi-agent support.

    Parameters
    ----------
    name     : human-readable identifier, e.g. 'Push(Box1,Pb,Door2)'
    agent    : the agent that owns / executes this action
    preconds : preconditions (positive literals)
    add_list : positive effects
    del_list : negative effects (literals that become False)
    """
    _id_counter = count(1)

    def __init__(self, name, agent, preconds, add_list, del_list):
        self.name      = name
        self.agent     = agent
        self.preconds  = set(preconds)
        self.add_list  = set(add_list)
        self.del_list  = set(del_list)
        self.step_id   = None          # assigned when added to a plan

    def __repr__(self):
        sid = f"S{self.step_id:04d}" if self.step_id is not None else "S????"
        return f"[{self.agent}] {sid}: {self.name}"

    def __eq__(self, other):
        return isinstance(other, Action) and self.step_id == other.step_id

    def __hash__(self):
        return hash(self.step_id)


# ─── CausalLink ──────────────────────────────────────────────────────────────
class CausalLink:
    """
    AIMA causal link  producer ─(literal)→ consumer.

    Asserts that *literal* must remain True from *producer* until *consumer*.
    Any action deleting *literal* that is not safely ordered is a **threat**.
    """
    def __init__(self, producer, literal, consumer):
        self.producer = producer
        self.literal  = literal
        self.consumer = consumer

    def __repr__(self):
        return (f"{self.producer.name}(S{self.producer.step_id})"
                f" --({self.literal})--> "
                f"{self.consumer.name}(S{self.consumer.step_id})")


# ─── POPlan ──────────────────────────────────────────────────────────────────
class POPlan:
    """
    The AIMA partial-order plan: steps, orderings, causal_links, agenda.

    Dummy actions START and FINISH are always present:
      START.add_list  = initial_state   (no preconds)
      FINISH.preconds = goal_state      (no effects)
    """

    def __init__(self, start: "Action", finish: "Action"):
        self.start   = start
        self.finish  = finish
        self.steps   = [start, finish]
        self.orderings: list[tuple] = [(start.step_id, finish.step_id)]
        self.causal_links: list[CausalLink] = []
        self.agenda: list[tuple] = [(finish, p) for p in finish.preconds]

    # ── Ordering helpers ──────────────────────────────────────────────────────
    def add_ordering(self, a: "Action", b: "Action") -> bool:
        pair = (a.step_id, b.step_id)
        if pair not in self.orderings:
            self.orderings.append(pair)
            return True
        return False

    def is_before(self, a_id: int, b_id: int) -> bool:
        """Transitive closure: is step a_id ordered before b_id?"""
        if a_id == b_id:
            return False
        visited, queue = set(), deque([a_id])
        while queue:
            curr = queue.popleft()
            if curr == b_id:
                return True
            if curr not in visited:
                visited.add(curr)
                queue.extend(y for (x, y) in self.orderings if x == curr)
        return False

    def would_be_cyclic(self, a: "Action", b: "Action") -> bool:
        return self.is_before(b.step_id, a.step_id)

    # ── Step management ───────────────────────────────────────────────────────
    def add_step(self, action: "Action", before: "Action", after: "Action"):
        """
        Insert *action* with orderings  before ≺ action ≺ after.
        Enqueues action's preconditions onto the agenda.
        """
        action.step_id = next(Action._id_counter)
        self.steps.append(action)
        self.add_ordering(before, action)
        self.add_ordering(action, after)
        for p in action.preconds:
            self.agenda.append((action, p))
        return action

    # ── Causal link management ────────────────────────────────────────────────
    def add_causal_link(self, producer: "Action", literal: str,
                        consumer: "Action") -> CausalLink:
        cl = CausalLink(producer, literal, consumer)
        self.causal_links.append(cl)
        return cl

    # ── Threat detection & resolution (AIMA: promotion / demotion) ────────────
    def threatens(self, step: "Action", cl: CausalLink) -> bool:
        """
        AIMA §11.2 threat condition:
          step threatens cl  iff
            • step ≠ producer  and  step ≠ consumer
            • step.del_list contains cl.literal
            • step is NOT already ordered after consumer  (not promoted)
            • step is NOT already ordered before producer (not demoted)
        """
        if step is cl.producer or step is cl.consumer:
            return False
        if cl.literal not in step.del_list:
            return False
        if self.is_before(cl.consumer.step_id, step.step_id):
            return False
        if self.is_before(step.step_id, cl.producer.step_id):
            return False
        return True

    def resolve_threat(self, threat: "Action", cl: CausalLink) -> bool:
        """
        Try promotion (threat after consumer), then demotion (threat before producer).
        Returns True if resolved, False if the plan is stuck.
        """
        if not self.would_be_cyclic(cl.consumer, threat):
            self.add_ordering(cl.consumer, threat)
            return True
        if not self.would_be_cyclic(threat, cl.producer):
            self.add_ordering(threat, cl.producer)
            return True
        return False

    # ── Topological linearization ─────────────────────────────────────────────
    def topological_order(self) -> list:
        """
        One valid total order consistent with the partial-order constraints.
        Excludes START and FINISH (dummy steps).
        """
        id_to_step = {s.step_id: s for s in self.steps}
        in_deg  = defaultdict(int)
        graph   = defaultdict(list)
        for (a, b) in self.orderings:
            graph[a].append(b)
            in_deg[b] += 1
        queue  = deque([s for s in self.steps if in_deg[s.step_id] == 0])
        result = []
        while queue:
            node = queue.popleft()
            result.append(node)
            for nxt_id in graph[node.step_id]:
                in_deg[nxt_id] -= 1
                if in_deg[nxt_id] == 0:
                    queue.append(id_to_step[nxt_id])
        return [s for s in result if s.name not in ("START", "FINISH")]


print("✓ Action, CausalLink, POPlan defined")


✓ Action, CausalLink, POPlan defined


## 2. Partial-Order Planner — `pop_solve` (AIMA Algorithm 11.1)

In [2]:
def pop_solve(initial_state, goal_state, action_library,
              max_iter=500, verbose=True):
    """
    AIMA-style Partial-Order Planner (Algorithm 11.1).

    The algorithm (AIMA §11.2):
      1. Pick an open precondition (B, p) from the agenda.
      2. Choose a provider A  (existing step or new action from library).
         Preference order:
           a) existing step already in plan & ordered before B
           b) START (initial state)
           c) new action (least effects = fewest potential threats)
         KEY: when adding a new action we first check whether an existing
         step of the same name can be reused — this avoids the STRIPS
         'duplicate step' problem that causes false threats.
      3. Add causal link A ─p→ B.
         If A is new, add it with  START ≺ A ≺ B.
         If A is existing but not yet ordered before B, add A ≺ B.
      4. For every step S that now threatens the new causal link, call
         resolve_threat() (promotion / demotion).
      5. Repeat until agenda is empty → plan is a solution.

    Multi-agent extension:
      CoordAgent owns START and FINISH (planning authority).
      All physical actions belong to domain-specific agents (e.g. RobotAgent).

    Parameters
    ----------
    initial_state  : set of string literals
    goal_state     : set of string literals
    action_library : list of Action templates (will be deepcopied)
    max_iter       : safety limit on loop iterations
    verbose        : print resolution trace

    Returns  POPlan | None
    """
    # ── Dummy START and FINISH actions ──────────────────────────────────────
    start  = Action("START",  "CoordAgent", [], list(initial_state), [])
    finish = Action("FINISH", "CoordAgent", list(goal_state), [], [])
    start.step_id  = 0
    finish.step_id = 9999

    plan = POPlan(start, finish)

    for iteration in range(max_iter):
        if not plan.agenda:
            break

        # ── Step 1: pick open precondition ─────────────────────────────────
        goal_step, precond = plan.agenda.pop(0)
        if verbose:
            print(f"  [{iteration+1:03d}] Open precond: «{precond}»"
                  f"  needed by {goal_step.name}")

        # ── Step 2: find provider ──────────────────────────────────────────
        provider = None

        # 2a. Existing step already ordered before goal_step?
        for step in plan.steps:
            if (precond in step.add_list
                    and not plan.would_be_cyclic(step, goal_step)):
                if plan.is_before(step.step_id, goal_step.step_id):
                    provider = step
                    break

        # 2b. Can START supply it and is it not cyclic?
        if provider is None and precond in start.add_list:
            if not plan.would_be_cyclic(start, goal_step):
                provider = start
                plan.add_ordering(start, goal_step)

        # 2c. Reuse an existing step even if not yet ordered (add ordering)
        if provider is None:
            for step in plan.steps:
                if (step is not finish
                        and precond in step.add_list
                        and not plan.would_be_cyclic(step, goal_step)):
                    plan.add_ordering(step, goal_step)
                    provider = step
                    break

        # 2d. Add a brand-new action from the library
        if provider is None:
            candidates = [a for a in action_library if precond in a.add_list]
            if not candidates:
                if verbose:
                    print(f"       ✗ No action achieves «{precond}» — plan failure")
                return None
            # Least-commitment heuristic: fewest delete-effects → fewest threats
            candidates.sort(key=lambda a: len(a.del_list))
            new_action = deepcopy(candidates[0])
            plan.add_step(new_action, start, goal_step)
            provider = new_action
            if verbose:
                print(f"         + New step: {new_action.name}")

        # ── Step 3: add causal link ────────────────────────────────────────
        cl = plan.add_causal_link(provider, precond, goal_step)
        if verbose:
            print(f"         → provider: {provider.name}")
            print(f"           link    : {cl}")

        # ── Step 4: resolve threats ────────────────────────────────────────
        for step in list(plan.steps):
            if plan.threatens(step, cl):
                if verbose:
                    print(f"         ⚠ Threat: {step.name} on «{cl.literal}»")
                if not plan.resolve_threat(step, cl):
                    if verbose:
                        print(f"         ✗ Unresolvable threat — plan failure")
                    return None
                if verbose:
                    print(f"           ✓ Resolved")

    # ── Done ────────────────────────────────────────────────────────────────
    if plan.agenda:
        print("✗ Max iterations reached: unresolved preconditions remain.")
        return None

    if verbose:
        print("\n✓ Centralized POP — consistent plan found.")
    return plan


print("✓ pop_solve() defined")


✓ pop_solve() defined


## 3. Multi-Agent Distribution

Once the `CoordAgent` has the global POP, `distribute_plan` splits it by agent.  
**Inter-agent causal links** expose explicit dependencies between agents — crucial for  
coordination protocols in MAS.


In [3]:
def distribute_plan(plan: POPlan, verbose=True) -> dict:
    """
    Split global POP into per-agent ordered sub-plans.

    Returns  {agent_name: [ordered Action, ...]}
    """
    topo = plan.topological_order()
    agent_plans: dict = defaultdict(list)
    for step in topo:
        agent_plans[step.agent].append(step)
    result = dict(agent_plans)

    if verbose:
        print("=" * 64)
        print("  DISTRIBUTED EXECUTION PLANS")
        print("=" * 64)
        for agent, steps in result.items():
            role = "Planner" if agent == "CoordAgent" else "Executor"
            print(f"\n  ┌─ {agent} [{role}] " + "─" * max(0, 38 - len(agent)))
            if not steps:
                print("  │  (no physical actions — planning role only)")
            for i, s in enumerate(steps, 1):
                print(f"  │  {i}. {s.name}")
                print(f"  │     pre  : {sorted(s.preconds)}")
                print(f"  │     add  : {sorted(s.add_list)}")
                print(f"  │     del  : {sorted(s.del_list)}")
            print("  └" + "─" * 44)

        print("\n─── Inter-Agent Causal Links ─────────────────────────────")
        found = False
        for cl in plan.causal_links:
            if cl.producer.agent != cl.consumer.agent:
                print(f"  {cl.producer.agent}  →  {cl.consumer.agent}"
                      f"   via «{cl.literal}»")
                found = True
        if not found:
            print("  (all causal links are intra-agent in this scenario)")

    return result


print("✓ distribute_plan() defined")


✓ distribute_plan() defined


## 4. `display_plan` — AIMA-style output

In [4]:
def display_plan(plan: POPlan):
    """AIMA-style display: causal links, orderings, linearization."""
    if plan is None:
        print("No plan to display.")
        return

    topo = plan.topological_order()
    id_map = {s.step_id: s for s in plan.steps}

    print("\n──────────────────────────────────────────────────────────")
    print("  CAUSAL LINKS")
    print("──────────────────────────────────────────────────────────")
    for cl in plan.causal_links:
        print(f"  {cl}")

    print("\n──────────────────────────────────────────────────────────")
    print("  ORDERING CONSTRAINTS  (a ≺ b  =  a must come before b)")
    print("──────────────────────────────────────────────────────────")
    for (a, b) in sorted(plan.orderings):
        na = id_map[a].name if a in id_map else str(a)
        nb = id_map[b].name if b in id_map else str(b)
        print(f"  {na} (S{a:04d})  ≺  {nb} (S{b:04d})")

    print("\n──────────────────────────────────────────────────────────")
    print("  PARTIAL-ORDER PLAN  (one valid linearization via toposort)")
    print("──────────────────────────────────────────────────────────")
    print("  START")
    for s in topo:
        print(f"    → {s.name}  [{s.agent}]")
    print("  FINISH")
    print("──────────────────────────────────────────────────────────")


print("✓ display_plan() defined")


✓ display_plan() defined


## 5. Problem Definition — Robot World

> A robot must move a box from **Room 2** (position Pb) to **Room 1** (position Pg)  
> by navigating through a corridor.  The robot starts in **Room 3** (position Pr).

```
Room3 (Pr) ─── Door3 ─── Corridor ─── Door2 ─── Room2 (Pb)
                                   └── Door1 ─── Room1 (Pg)  ← goal
```

Two agents:
| Agent | Role | Actions |
|-------|------|---------|
| **CoordAgent** | Centralized planner | START, FINISH (virtual) |
| **RobotAgent** | Executor | Go(Robot,x,y), Push(b,x,y) |


In [5]:
initial_state = {
    "At(Robot,Pr)",
    "At(Box1,Pb)",
    "Pushable(Box1)",
    "In(Pr,Room3)",
    "In(Door3,Room3)", "In(Door3,Corridor)",
    "In(Door2,Room2)", "In(Door2,Corridor)",
    "In(Pb,Room2)",
    "In(Door1,Room1)", "In(Door1,Corridor)",
    "In(Pg,Room1)",
}

goal_state = {"At(Box1,Pg)"}

print("Initial state:")
for f in sorted(initial_state):
    print(f"  {f}")
print("\nGoal:", sorted(goal_state))


Initial state:
  At(Box1,Pb)
  At(Robot,Pr)
  In(Door1,Corridor)
  In(Door1,Room1)
  In(Door2,Corridor)
  In(Door2,Room2)
  In(Door3,Corridor)
  In(Door3,Room3)
  In(Pb,Room2)
  In(Pg,Room1)
  In(Pr,Room3)
  Pushable(Box1)

Goal: ['At(Box1,Pg)']


In [6]:
def build_action_library() -> list:
    """All ground Go and Push actions for the Robot World (belong to RobotAgent)."""
    lib = []

    # Go(Robot, from, to) ─ navigate between adjacent positions
    go_pairs = [
        ("Pr",    "Door3"),  ("Door3", "Pr"),
        ("Door3", "Door2"),  ("Door2", "Door3"),
        ("Door2", "Pb"),     ("Pb",    "Door2"),
        ("Door2", "Door1"),  ("Door1", "Door2"),
        ("Door1", "Pg"),     ("Pg",    "Door1"),
    ]
    for frm, to in go_pairs:
        lib.append(Action(
            name     = f"Go(Robot,{frm},{to})",
            agent    = "RobotAgent",
            preconds = [f"At(Robot,{frm})"],
            add_list = [f"At(Robot,{to})"],
            del_list = [f"At(Robot,{frm})"],
        ))

    # Push(box, from, to) ─ robot pushes box to adjacent position
    push_triples = [
        ("Box1", "Pb",    "Door2"),
        ("Box1", "Door2", "Door1"),
        ("Box1", "Door1", "Pg"),
    ]
    for box, frm, to in push_triples:
        lib.append(Action(
            name     = f"Push({box},{frm},{to})",
            agent    = "RobotAgent",
            preconds = [f"At(Robot,{frm})", f"At({box},{frm})", f"Pushable({box})"],
            add_list = [f"At(Robot,{to})", f"At({box},{to})"],
            del_list = [f"At(Robot,{frm})", f"At({box},{frm})"],
        ))

    return lib

action_library = build_action_library()
print(f"✓ {len(action_library)} actions in the library:")
for a in action_library:
    print(f"   {a.name}")


✓ 13 actions in the library:
   Go(Robot,Pr,Door3)
   Go(Robot,Door3,Pr)
   Go(Robot,Door3,Door2)
   Go(Robot,Door2,Door3)
   Go(Robot,Door2,Pb)
   Go(Robot,Pb,Door2)
   Go(Robot,Door2,Door1)
   Go(Robot,Door1,Door2)
   Go(Robot,Door1,Pg)
   Go(Robot,Pg,Door1)
   Push(Box1,Pb,Door2)
   Push(Box1,Door2,Door1)
   Push(Box1,Door1,Pg)


## 6. Run the Centralized POP (CoordAgent)

In [7]:
print("=" * 64)
print("  Centralized POP  —  CoordAgent planning")
print("=" * 64)

plan = pop_solve(initial_state, goal_state, action_library, verbose=True)


  Centralized POP  —  CoordAgent planning
  [001] Open precond: «At(Box1,Pg)»  needed by FINISH
         + New step: Push(Box1,Door1,Pg)
         → provider: Push(Box1,Door1,Pg)
           link    : Push(Box1,Door1,Pg)(S1) --(At(Box1,Pg))--> FINISH(S9999)
  [002] Open precond: «Pushable(Box1)»  needed by Push(Box1,Door1,Pg)
         → provider: START
           link    : START(S0) --(Pushable(Box1))--> Push(Box1,Door1,Pg)(S1)
  [003] Open precond: «At(Robot,Door1)»  needed by Push(Box1,Door1,Pg)
         + New step: Go(Robot,Door2,Door1)
         → provider: Go(Robot,Door2,Door1)
           link    : Go(Robot,Door2,Door1)(S2) --(At(Robot,Door1))--> Push(Box1,Door1,Pg)(S1)
  [004] Open precond: «At(Box1,Door1)»  needed by Push(Box1,Door1,Pg)
         + New step: Push(Box1,Door2,Door1)
         → provider: Push(Box1,Door2,Door1)
           link    : Push(Box1,Door2,Door1)(S3) --(At(Box1,Door1))--> Push(Box1,Door1,Pg)(S1)
  [005] Open precond: «At(Robot,Door2)»  needed by Go(Robot,Door2,D

## 7. Global Plan — Causal Links, Orderings, Linearization

In [8]:
display_plan(plan)

No plan to display.


## 8. Distribute the Plan to Agents

In [9]:
if plan:
    distributed = distribute_plan(plan, verbose=True)


## 9. Multi-Agent POP Summary

In [10]:
if plan:
    topo_all    = plan.topological_order()
    robot_steps = distributed.get("RobotAgent", [])

    print("╔══════════════════════════════════════════════════════════╗")
    print("║         MULTI-AGENT POP SUMMARY                         ║")
    print("╠══════════════════════════════════════════════════════════╣")
    print(f"║  Action steps (excl START/FINISH) : {len(topo_all):>3d}               ║")
    print(f"║  Causal links established         : {len(plan.causal_links):>3d}               ║")
    print(f"║  Ordering constraints             : {len(plan.orderings):>3d}               ║")
    print("╠══════════════════════════════════════════════════════════╣")
    print("║  Architecture : Centralized Planning / Distributed Exec ║")
    print("║  Paradigm     : MAS Ch. 3  •  AIMA §11.2               ║")
    print("╠══════════════════════════════════════════════════════════╣")
    print("║  CoordAgent  : computed the POP globally                ║")
    print("║  RobotAgent  : executes its sub-plan autonomously       ║")
    print("╠══════════════════════════════════════════════════════════╣")
    print("║  Execution order for RobotAgent:                        ║")
    for i, s in enumerate(robot_steps, 1):
        line = f"║    {i}. {s.name}"
        print(line.ljust(58) + "║")
    print("╚══════════════════════════════════════════════════════════╝")


## 10. Bonus — AIMA Classic: Spare-Tire Problem

The canonical AIMA POP benchmark (§11.2).  
Validates the planner on the textbook example.


In [11]:
spare_tire_lib = [
    Action(
        name     = "Remove(Flat,Axle)",
        agent    = "Mechanic",
        preconds = ["On(Flat,Axle)"],
        add_list = ["At(Flat,Ground)"],
        del_list = ["On(Flat,Axle)"],
    ),
    Action(
        name     = "Remove(Spare,Trunk)",
        agent    = "Mechanic",
        preconds = ["In(Spare,Trunk)"],
        add_list = ["At(Spare,Ground)"],
        del_list = ["In(Spare,Trunk)"],
    ),
    Action(
        name     = "PutOn(Spare,Axle)",
        agent    = "Mechanic",
        preconds = ["At(Spare,Ground)", "At(Flat,Ground)"],
        add_list = ["On(Spare,Axle)"],
        del_list = ["At(Spare,Ground)"],
    ),
    Action(
        name     = "LeaveOvernight",
        agent    = "Adversary",
        preconds = [],
        add_list = [],
        del_list = ["At(Spare,Ground)", "At(Flat,Ground)",
                    "On(Flat,Axle)",    "In(Spare,Trunk)"],
    ),
]

spare_init = {"On(Flat,Axle)", "In(Spare,Trunk)"}
spare_goal = {"On(Spare,Axle)", "At(Flat,Ground)"}

print("=" * 64)
print("  AIMA Spare-Tire Problem")
print("=" * 64)
st_plan = pop_solve(spare_init, spare_goal, spare_tire_lib, verbose=True)
display_plan(st_plan)


  AIMA Spare-Tire Problem
  [001] Open precond: «At(Flat,Ground)»  needed by FINISH
         + New step: Remove(Flat,Axle)
         → provider: Remove(Flat,Axle)
           link    : Remove(Flat,Axle)(S6) --(At(Flat,Ground))--> FINISH(S9999)
  [002] Open precond: «On(Spare,Axle)»  needed by FINISH
         + New step: PutOn(Spare,Axle)
         → provider: PutOn(Spare,Axle)
           link    : PutOn(Spare,Axle)(S7) --(On(Spare,Axle))--> FINISH(S9999)
  [003] Open precond: «On(Flat,Axle)»  needed by Remove(Flat,Axle)
         → provider: START
           link    : START(S0) --(On(Flat,Axle))--> Remove(Flat,Axle)(S6)
  [004] Open precond: «At(Spare,Ground)»  needed by PutOn(Spare,Axle)
         + New step: Remove(Spare,Trunk)
         → provider: Remove(Spare,Trunk)
           link    : Remove(Spare,Trunk)(S8) --(At(Spare,Ground))--> PutOn(Spare,Axle)(S7)
  [005] Open precond: «At(Flat,Ground)»  needed by PutOn(Spare,Axle)
         → provider: Remove(Flat,Axle)
           link    : Remo

## 11. Bonus — AIMA Classic: Socks and Shoes

Another textbook POP example.  Demonstrates parallel sub-plans (left / right legs)  
with no ordering constraint between them.


In [12]:
ss_lib = [
    Action("RightShoe",  "Person", ["RightSockOn"], ["RightShoeOn"], []),
    Action("RightSock",  "Person", [],               ["RightSockOn"], []),
    Action("LeftShoe",   "Person", ["LeftSockOn"],  ["LeftShoeOn"],  []),
    Action("LeftSock",   "Person", [],               ["LeftSockOn"],  []),
]

ss_init = set()
ss_goal = {"RightShoeOn", "LeftShoeOn"}

print("=" * 64)
print("  AIMA Socks-and-Shoes Problem")
print("=" * 64)
ss_plan = pop_solve(ss_init, ss_goal, ss_lib, verbose=True)
display_plan(ss_plan)


  AIMA Socks-and-Shoes Problem
  [001] Open precond: «RightShoeOn»  needed by FINISH
         + New step: RightShoe
         → provider: RightShoe
           link    : RightShoe(S9) --(RightShoeOn)--> FINISH(S9999)
  [002] Open precond: «LeftShoeOn»  needed by FINISH
         + New step: LeftShoe
         → provider: LeftShoe
           link    : LeftShoe(S10) --(LeftShoeOn)--> FINISH(S9999)
  [003] Open precond: «RightSockOn»  needed by RightShoe
         + New step: RightSock
         → provider: RightSock
           link    : RightSock(S11) --(RightSockOn)--> RightShoe(S9)
  [004] Open precond: «LeftSockOn»  needed by LeftShoe
         + New step: LeftSock
         → provider: LeftSock
           link    : LeftSock(S12) --(LeftSockOn)--> LeftShoe(S10)

✓ Centralized POP — consistent plan found.

──────────────────────────────────────────────────────────
  CAUSAL LINKS
──────────────────────────────────────────────────────────
  RightShoe(S9) --(RightShoeOn)--> FINISH(S9999)
  LeftSh